# 参数校准与恢复

独立合成 sandbox，无设备、无前置课程。完成一次“采集→拟合候选→重新采集验证→发布到参数分支”。这不是自动 freshness 或真实传感器物理模型。所有模型和策略都在 `src/my_experiment/calibration.py`，可直接阅读修改。

先顺序运行。每次完整重跑创建一个新的练习分支，已有结果保留；需要全新目录时使用教学入口的重置。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
from uuid import uuid4
from my_experiment.calibration import Sensor, CalibrationIntent, calibrate

assert session.project_root is not None
project = sc.open_project(session.project_root)
lab = project.connect()
trial = "zero-" + uuid4().hex[:8]
initial = lab.parameters.save(
    name=trial + "-initial",
    catalog=sc.parameter_catalog("zero", Sensor),
    parameters=sc.parameter_snapshot("zero-inputs", tables={Sensor: [Sensor(id="q0", offset=0.1)]}),
)
destination = lab.parameters.create_branch(trial, revision=initial)
print("本次参数分支:", trial)

## 先保留候选，不发布

合成模型的零点是 0.24，初始设置是 0.1。拟合用测量残差修正设置；它不直接修改参数分支。请求捕获参数、setup 和目标分支版本，恢复时不会重新读取新的分支头。下面故意在完成 baseline 和 fit 后暂停。

In [ ]:
intent = CalibrationIntent(
    initial=lab.parameters.resolve(initial), destination=destination,
    revision_name=trial + "-accepted",
)
request = lab.procedures.submit(calibrate, intent, request_key=trial)
lab.procedures.resume_snapshot(
    request.snapshot, should_yield=lambda: len(request.steps().items) >= 2,
)
request_id = request.id
assert request.state == "ready"
assert lab.parameters.checkout(trial).head == destination
request.summary()

## 断开后恢复

这里只关闭 Notebook 客户端连接。daemon 重启后的恢复由框架回归覆盖；实机断电不在本课范围。恢复必须使用原请求，不能把新分支头塞回旧意图。已有步骤重放记录，不重复采集。

In [ ]:
lab.close()
lab = project.connect()
request = lab.procedures.get(request_id)
request.resume()
assert request.summary().outcome == "succeeded"
accepted = lab.parameters.checkout(trial).head
assert accepted.generation == destination.generation + 1
assert len(request.steps().items) == 5
request.summary()

验证使用候选参数重新采集，并额外施加 0.002 的扰动；残差在阈值内且比原值小时才接受。`publish` 保存证据并推进这个分支，setup 和共享默认值不会随之改变。手动 `params.save()` 只是保存编辑，不会生成这条验证证据链。

In [ ]:
print("接受后的参数版本:", accepted.revision)
print("步骤记录:", request.steps())
# 重复恢复已完成请求不会再次执行测量或推进分支。
request.resume()
assert lab.parameters.checkout(trial).head == accepted

## 验证失败是什么样

在新分支上把独立验证扰动设为 0.1，保留失败决策，但不发布。真实实验的阈值、模型和所需测量由实验室决定。

In [ ]:
rejected_branch = lab.parameters.create_branch(trial + "-reject", revision=initial)
rejected = lab.procedures.submit(
    calibrate,
    CalibrationIntent(
        initial=lab.parameters.resolve(initial), destination=rejected_branch,
        revision_name=trial + "-not-published", disturbance=0.1,
    ),
    request_key=trial + "-reject",
)
try:
    rejected.resume()
except ValueError as error:
    print(error)
assert rejected.summary().outcome == "failed"
assert lab.parameters.checkout(trial + "-reject").head == rejected_branch
rejected.summary()

In [ ]:
verification = lab.published_analysis(rejected.output("verify").analysis_record_id)
decision = verification.fact("decision").value
assert decision["accepted"] is False
decision

## 日常检查不必修改参数

下面对已接受的精确参数版本分别做一次正常检查和一次漂移检查。两次流程都成功完成，但只有前者指标达标。检查只保留采集与分析证据，不产生候选或新的参数版本。重复 resume 是恢复原请求；想获得新的检查证据，需要新请求。

In [ ]:
from my_experiment.calibration import CHECK_RESULT, CheckIntent, check_zero
from scopecat.automation.calibration import CheckEvidence

check_history = []
for label, disturbance, expected in (("healthy", 0.002, True), ("drifted", 0.1, False)):
    health = lab.procedures.submit(
        check_zero,
        CheckIntent(initial=lab.parameters.resolve(accepted.revision), disturbance=disturbance),
        request_key=trial + "-" + label,
    )
    health.resume()
    assert health.summary().outcome == "succeeded"
    assert len(health.steps().items) == 2
    measured = lab.get_run(health.output("check").run_id)
    report = measured.published_analysis(health.output("assess").analysis_record_id)
    assert not report.parameter_proposals
    result = report.fact_as("check", CHECK_RESULT)
    assert result.passed is expected
    check_history.append(CheckEvidence(measured.snapshot, result.scope, report.id, result.passed))
    health.resume()
    assert health.output("assess").analysis_record_id == report.id
    assert lab.parameters.checkout(trial).head == accepted
    print(label, result)


## 证据何时仍适用

下面明确要求与刚才相同的模拟场景、对象和 setup，使用已接受参数版本。真实维护调用者应从当前选择的测量上下文构造请求，不能自动把历史记录的上下文当成当前上下文。

第一版只允许精确版本复用；改阈值、条件或参数版本会要求重查。时间保守地从 run 创建时计算，重新分析不会让旧数据变新。这是供策略使用的判断，不会自动发布或修复。

In [ ]:
from dataclasses import replace
from datetime import datetime, timedelta, timezone
from my_experiment.calibration import check_scope
from scopecat.automation.calibration import CalibrationContext, assess_calibration_check, select_calibration_check

observed = measured.snapshot
binding = observed.scientific_binding
current = CalibrationContext(accepted.revision, binding.subject, binding.setup_content_hash, binding.scenario)
now = datetime.now(timezone.utc)
def assess(scope=check_scope(0.01), context=current, at=now):
    return assess_calibration_check(
        observed, checked_scope=result.scope, requested_scope=scope, passed=result.passed,
        current=context, now=at, max_age=timedelta(hours=1),
    )

# 上一段最后一次检查测得漂移；执行成功不代表能力可用。
assert assess().status == "out_of_spec"
assert "policy_changed" in assess(check_scope(0.02)).reasons
assert "parameters_changed" in assess(context=replace(current, parameters=destination.revision)).reasons
expired = assess(at=observed.created_at + timedelta(hours=1))
assert expired.status == "recheck"
assert "check_expired" in expired.reasons
print(assess(), expired)

# 这里提供本次练习两次检查的完整历史；顺序不影响选择。
selected = select_calibration_check(
    tuple(reversed(check_history)), requested_scope=check_scope(0.01), current=current,
    now=now, max_age=timedelta(hours=1), history_complete=True,
)
assert selected.status == "out_of_spec"
assert selected.evidence.analysis_record_id == report.id
print(selected.reason, selected.assessment)


较新的失败检查覆盖了旧的通过检查，不能为获得通过结果而回退。历史分页不完整时必须传 `history_complete=False`；未完成的检查也必须包含在历史里。选择器不扫描数据库。同一次测量的不同重分析若没有明确取舍，结果为歧义；重新分析不会刷新测量时间。

这里没有诊断漂移原因或自动修复：这些需要明确的维护策略。

## 自己修改与重开

- 打开 `calibration.py`，先读 `measure_zero`、`fit_zero` 和 `verify_zero`，最后读 `calibrate`。采集、策略和持久化步骤的区别就在这里。
- 改意图中的 `tolerance` 或 `disturbance`，从新练习分支提交新请求。修改 procedure 源码时升级 `version`，重新连接后提交新请求；不要期待普通实验的 live refresh 自动替换正在恢复的 procedure。
- 重启 kernel 后连接 `lab = sc.open_project(session.project_root).connect()`，用 `lab.procedures.list()` 查看历史，再 `lab.procedures.get(选中的请求.id)`；不必维护记录 ID 的 JSON 文件。
- 如遇 `attention_required`，先看原因；只有结果未知且仍使用原意图时才调用 `retry_attention()`。科学拒绝和分支已改变需要新的请求，不能强行重试发布。
- 多目标需要合并候选后重新验证联合结果，不能把每个目标单独通过等同于联合通过。见 public 的参数校准自动化指南。

完成后关闭连接；练习目录按 sandbox 生命周期保留，重置不会覆盖它。

In [ ]:
lab.close()